In [1]:
import copy

In [2]:
#Creating the Graph
G = Graph(12)

G.add_edges([(0,1),(1,2),(2,3),(1,3),(0,3)])

G.add_edges([(0,4), (1,5), (1,6), (1,7), (2,8), (3,9), (3,10), (3,11)])

In [3]:
#Labelling the half-edges
G.set_edge_label(0,4, (0,0))
G.set_edge_label(1,5,(1,0))
G.set_edge_label(1,6,(1,1))
G.set_edge_label(1,7,(1,2))
G.set_edge_label(2,8,(2,0))
G.set_edge_label(3,9,(3,0))
G.set_edge_label(3,10,(3,1))
G.set_edge_label(3,11,(3,2))

In [4]:
num_colors=6
#coloring = ((0,), (0, 1, 2), (0,), (0, 1, 2))

In [5]:
#!/usr/bin/env python

import itertools

def does_extend(G, coloring, num_colors):
    p = MixedIntegerLinearProgram()
    x= p.new_variable(binary=True)
    L= G.line_graph()
    for e in G.edges(): 
        p.add_constraint(p.sum(x[e,i] for i in range(num_colors)) == 1) #every edge must have one color
    
    for e,f in L.edges(labels=False):
        for i in range(num_colors):
            p.add_constraint(x[e,i] + x[f,i] <=1) #adjacent edges must have different colors
    
    for e in G.edges():
        if e[2]: #is not none
            tuple_index,color_index = e[2]
            p.add_constraint(x[e, coloring[tuple_index][color_index] ] == 1) #considers the established coloring of the half-edges
    
    p.set_objective(0)
    try:
        p.solve()
        return True
    except sage.numerical.mip.MIPSolverException:
        #print(coloring)
        return False
    
def test_colorings():
    
    num_colors=6
    C=list(range(num_colors))  # list of colors, for convenience
    degrees=[1,3,1,3]  # this would come from the graph
    
    S=set()  # the set of colorings that can be modified to an extendible coloring
    U=set()
    for coloring in itertools.product(
        *(
          #(frozenset(x) for x in itertools.combinations(C,d))  # generator
          (tuple(x) for x in itertools.combinations(C,d))
          for d in degrees
         )
        ):
        if does_extend(G, coloring, num_colors):  #extend conditions
            S.add(coloring)
        else:
            U.add(coloring)
        #print(coloring)
    print(len(S), len(U))
    return S,U
    
S,U = test_colorings()

12120 2280


In [6]:
S1 = set() #list that does work after one change
U1 = set() #list that don't work

In [7]:
def perform_Kempe(coloring, i, j):
    aux_edges = set()
    vertex_index = set()
    degree_two_vertices = set()
    
    # for trailing off Kempe Chains or loops in auxillary graph
    for k in range(len(coloring)):
        coloring_vertex = copy.deepcopy(list(coloring[k]))
        isChanged = 0
        for l in range(len(coloring_vertex)):
            test_coloring = copy.deepcopy(coloring)
            if coloring_vertex[l] == i:
                coloring_vertex[l] =j
                isChanged += 1
            else:
                if coloring_vertex[l] == j:
                    coloring_vertex[l]=i
                    isChanged += 1
        coloring_vertex.sort()
        if isChanged==1:
            test_coloring[k] = tuple(coloring_vertex)
            test_coloring = tuple(test_coloring)
            if test_coloring in S:
                S1.add(test_coloring)
            else:
                U1.add(test_coloring)
                aux_edges.add(tuple((k,)))
        if isChanged==2:
            test_coloring[k] = tuple(coloring_vertex)
            test_coloring = tuple(test_coloring)
            U1.add(test_coloring)
            aux_edges.add(tuple((k,)))
            
    #gets the set of vertices that contains the colors i and j and the "half-edge degree" it has
    for k in range(len(coloring)):
        color_index=set()
        degree =0
        for l in range(len(coloring[k])):
            if coloring[k][l] == i:
                color_index.add(l)
                degree+=1
            if coloring[k][l] == j:
                color_index.add(l)
                degree+=1
        if color_index != set():
            vertex_index.add(k)
            if degree == 2:
                degree_two_vertices.add(k)

    #testing the Kempe Chains to see if it extends (multiple vertices)
    for endpoints in itertools.combinations(vertex_index, 2):
        testing_vertex_index = copy.deepcopy(vertex_index)
        start = endpoints[0]
        end = endpoints[1]
        test_coloring = copy.deepcopy(coloring)
        coloring_s = copy.deepcopy(list(coloring[start]))
        coloring_e = copy.deepcopy(list(coloring[end]))
        for m in range(len(coloring_s)):
            if coloring_s[m] == i:
                coloring_s[m] =j
            else:
                if coloring_s[m] == j:
                    coloring_s[m]= i
        coloring_s.sort()
        for m in range(len(coloring_e)):
            if coloring_e[m] == i:
                coloring_e[m] =j
            else:
                if coloring_e[m] == j:
                    coloring_e[m]= i
        coloring_e.sort()
        test_coloring[start] = tuple(coloring_s)
        test_coloring[end] = tuple(coloring_e)
        testing_vertex_index.remove(endpoints[0])
        testing_vertex_index.remove(endpoints[1])
        
        if tuple(test_coloring) in S:
            S1.add(tuple(test_coloring))
        else:
            for middle in powerset(degree_two_vertices-set(endpoints)):
                kempe = set(middle) | set(endpoints)
                kempe= tuple(sorted(kempe))
                U1.add(tuple(test_coloring))
                aux_edges.add(kempe)
    return aux_edges

In [8]:
def check_Crossing(edge1, edge2): 
    is_crossing = True
    if len(edge1) == 1 or len(edge2)==1: #trailing off Kempe chains are not crossing
        is_crossing = False
    else:
        minimum = min(edge1[0], edge2[0]) #edges of Kempe are in order so just need to test the 1st values for the min
        if minimum in edge1:
            start_edge = edge1
            test_edge = set(edge2)
        else:
            start_edge = edge2
            test_edge = set(edge1)
        for k in range(len(start_edge)-1):
            test_range = set(range(start_edge[k], start_edge[k+1]+1))
            intersection = test_range & test_edge
            if intersection == test_edge:
                is_crossing = False
            
    return is_crossing

In [19]:
def check_Matching(aux, num_vertices):
    p = MixedIntegerLinearProgram()
    x= p.new_variable(binary=True)
    matching = set()

    if len(aux) != 0 or len(aux) != 1:
        for i in range(len(aux)):
            for j in range(len(aux)):
                if i>j:
                    crossing = check_Crossing(aux[i], aux[j]) #check crossing
                    if crossing == True:
                        print(f'crossing between {aux[i]} and {aux[j]}')
                        p.add_constraint(x[i]+x[j] <= 1)
                    
    #constraints to make sure all vertices are covered
    for i in range(num_vertices):
        adj_edges = set()
        for j in range(len(aux)):
            if i in aux[j]:
                adj_edges.add(aux[j])
        if adj_edges != set():
            p.add_constraint(p.sum(x[j] for j in adj_edges) ==1)
            
    p.set_objective(0)
    try:
        p.solve()
        return True
    except sage.numerical.mip.MIPSolverException:
        #print(coloring)
        return False

In [20]:
def test_all_colors(coloring):
    CL = copy.deepcopy(list(coloring))
    for i in range(num_colors):
        for j in range(num_colors):
            if i < j:
                aux=tuple(perform_Kempe(CL, i, j))
                print(f'{aux} for {i} and {j}')
                matching = check_Matching(aux,4)
                if matching == True:
                    print(f'Found a non-crossing perfect matching auxilliary for colors {i} and {j}')
                #else:
                    #print(f'No non-crossing perfect matching found')
    #return aux_edges

In [21]:
UL = list(U)
test_all_colors(UL[3])
print(UL[3])

((1,), (3,), (1, 3)) for 0 and 1
Found a non-crossing perfect matching auxilliary for colors 0 and 1
((3,),) for 0 and 2
Found a non-crossing perfect matching auxilliary for colors 0 and 2
((1,), (3,), (1, 3)) for 0 and 3
Found a non-crossing perfect matching auxilliary for colors 0 and 3
((1,), (3,), (1, 3)) for 0 and 4
Found a non-crossing perfect matching auxilliary for colors 0 and 4
((1,), (3,), (1, 3)) for 0 and 5
Found a non-crossing perfect matching auxilliary for colors 0 and 5
((0, 2), (1, 3)) for 1 and 2
crossing between (1, 3) and (0, 2)
Found a non-crossing perfect matching auxilliary for colors 1 and 2
((1,),) for 1 and 3
Found a non-crossing perfect matching auxilliary for colors 1 and 3
((1,),) for 1 and 4
Found a non-crossing perfect matching auxilliary for colors 1 and 4
((1,), (3,), (1, 3)) for 1 and 5
Found a non-crossing perfect matching auxilliary for colors 1 and 5
((0, 2), (1, 3)) for 2 and 3
crossing between (1, 3) and (0, 2)
Found a non-crossing perfect matchi

In [13]:
print(UL[3])

((2,), (0, 1, 3), (2,), (0, 2, 5))


In [18]:
q = MixedIntegerLinearProgram()
x= q.new_variable(binary=True)
matching = set()
coloring = list(UL[3])
aux = tuple(perform_Kempe(coloring, 0,1))
print(aux)
num_vertices = 4

if len(aux) != 0 or len(aux) != 1:
    for i in range(len(aux)):
        for j in range(len(aux)):
            if i>j:
                crossing = check_Crossing(aux[i], aux[j]) #check crossing
                if crossing == True:
                    print(f'crossing between {aux[i]} and {aux[j]}')
                    p.add_constraint(x[i]+x[j] <= 1)
                        
for i in range(num_vertices+1):
    adj_edges = set()
    for j in range(len(aux)):
        if i in aux[j]:
            adj_edges.add(aux[j])
    if adj_edges != set():
        q.add_constraint(q.sum(x[j] for j in adj_edges) ==1)
            
q.set_objective(0)
q.show()

((1,), (3,), (1, 3))
Constraints:
  1.0 <= x_0 + x_1 <= 1.0
  1.0 <= x_1 + x_2 <= 1.0
Variables:
  x_0 is a boolean variable (min=0.0, max=1.0)
  x_1 is a boolean variable (min=0.0, max=1.0)
  x_2 is a boolean variable (min=0.0, max=1.0)


In [ ]:
for i,coloring in enumerate(UL):
    if coloring[0][0] != coloring[2][0]:
        if coloring[0][0] in coloring[1] and coloring[0][0] in coloring[3] and coloring[2][0] in coloring[1] and coloring[2][0] in coloring[3]:
            print(i,coloring)

In [ ]:
print(S1)

In [ ]:
UL.index(((5,), (0, 2, 3), (5,), (0, 4, 5)))
UL[122]
if testing_vertex_index is not set():
            for n in range(len(testing_vertex_index)):
                test_coloring_middle = copy.deepcopy(test_coloring)
                middle = testing_vertex_index[n][0] ##error here
                coloring_m = copy.deepcopy(list(coloring[middle]))
                for m in range(len(coloring_m)):
                    if coloring_m[m] == i:
                        coloring_m[m] =j
                    else:
                        if coloring_m[m] == j:
                            coloring_m[m]= i
                coloring_m.sort()
                test_coloring_middle[middle] = tuple(coloring_m)
                test_coloring_middle=tuple(test_coloring_middle)
                if test_coloring_middle in S:
                    S1.add(test_coloring_middle)
                else:
                    U1.add(test_coloring_middle)
                    aux_edges.add(tuple((start, middle, end)))   

In [48]:
# making the Mixed Integer Linear Program Constraints
p = MixedIntegerLinearProgram()
x= p.new_variable(binary=True)
L= G.line_graph()
for e in G.edges(): 
    p.add_constraint(p.sum(x[e,i] for i in range(num_colors)) == 1) 

for e,f in L.edges(labels=False):
    for i in range(num_colors):
        p.add_constraint(x[e,i] + x[f,i] <=1)

for e in G.edges():
    if e[2]: #is not none
        tuple_index,color_index = e[2]
        p.add_constraint(x[e, coloring[tuple_index][color_index] ] == 1)

p.set_objective(0)
p.show()        

NameError: name 'coloring' is not defined

In [49]:
print('Objective Value: {}'.format(p.solve()))

Objective Value: 0.0


In [47]:
for e in G.edges():
    for i in range(num_colors):
        if p.get_values(x[e,i], convert = ZZ, tolerance = 1e-3) == 1:
            print(f'x[{e}, {i}] = 1')

NameError: name 'p' is not defined